In [ ]:
# MDS of all ordered pairs (b1, from CSV) at all requested steps, with Kruskal stress-1 and (raw) Kruskal stress
import numpy as np
import matplotlib.pyplot as plt
from sklearn import manifold
from sklearn.metrics import euclidean_distances
from scipy.stats import pearsonr
import pandas as pd

# Use CSV-loaded b1 data if available; else load from gz in this folder
if 'all_b1s' in globals():
    B1_avg = all_b1s  # shape: (time_steps, items_n, b1_size) already averaged across seeds
else:
    def load_b1_mean_from_csv_gz(csv_gz_path):
        df = pd.read_csv(csv_gz_path, compression='gzip')
        seed_cols = [c for c in df.columns if c.startswith('seed_')]
        # Mean across seeds per (time_step, item, unit)
        grouped = df.groupby(['time_step', 'item', 'unit'])[seed_cols].mean()
        mean_series = grouped.mean(axis=1)  # average across seed columns
        time_steps = int(df['time_step'].max()) + 1
        items_n_inferred = int(df['item'].max()) + 1
        units = int(df['unit'].max()) + 1
        arr = np.zeros((time_steps, items_n_inferred, units))
        for (t, i, u), val in mean_series.items():
            arr[int(t), int(i), int(u)] = float(val)
        return arr
    # Try local gz produced earlier in this notebook
    B1_avg = load_b1_mean_from_csv_gz('conjunctive_lazy_rich_b1s.csv.gz')

time_steps, items_n_inferred, b1_size = B1_avg.shape

# Positions to display, used previously
positions = [0.00, 0.10, 0.20, 0.50, 0.65, 0.75, 0.85, 0.90, 1.00]
steps_after_training = time_steps - 1
step_idxs = [max(0, int(p * steps_after_training)) for p in positions]

# Utility to build all concatenated ordered pairs (i!=j)
def build_pairs(X_items):
    vecs, labels, pairs = [], [], []
    for i in range(items_n_inferred):
        for j in range(items_n_inferred):
            if i == j:
                continue
            v = np.concatenate([X_items[i], X_items[j]], axis=0)
            vecs.append(v)
            labels.append(f"({i},{j})")
            pairs.append((i, j))
    V = np.asarray(vecs)
    # mean-center features in each matrix for each step, stabilizes MDS
    V = V - V.mean(axis=0, keepdims=True)
    return V, labels, pairs

# Fit 2D metric MDS on each distance matrix; report Kruskal stress-1 and Kruskal stress (raw)
def fit_mds_with_stress(D, seed):
    mds = manifold.MDS(
        n_components=2,
        dissimilarity='precomputed',
        random_state=seed,
        n_init=4,
        normalized_stress='auto',
    )
    fit_result = mds.fit(D)
    coords = fit_result.embedding_
    stress_raw = getattr(mds, 'stress_', None)
    denom = float(np.sum(D ** 2))
    # Kruskal's stress-1 as in sklearn docs: sqrt( sum((d_ij - d̂_ij)^2) / sum(d_ij^2) )
    stress1 = np.sqrt(stress_raw / denom) if (stress_raw is not None and denom > 0) else np.nan

    # Compute the reconstructed distances
    Dr = euclidean_distances(coords)
    # Use only upper triangle (excluding diagonal)
    orig = D[np.triu_indices_from(D, k=1)]
    rec = Dr[np.triu_indices_from(Dr, k=1)]
    # Kruskal's raw stress
    kruskal_stress = np.sqrt(np.sum((orig - rec) ** 2) / np.sum(orig ** 2)) if np.sum(orig ** 2) > 0 else np.nan

    # For reporting: also include Pearson r between dist matrices
    if len(orig) > 1:
        r, _ = pearsonr(orig, rec)
    else:
        r = np.nan

    return coords, stress1, kruskal_stress, r

seed = mds_seed if 'mds_seed' in globals() else 0

nsteps = len(step_idxs)
ncols = 3
nrows = int(np.ceil(nsteps / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(5.5*ncols, 5*nrows), constrained_layout=True)

for pidx, (pos, step_idx) in enumerate(zip(positions, step_idxs)):
    V, labels, pairs = build_pairs(B1_avg[step_idx])
    D = euclidean_distances(V)
    coords, stress1, kst, r = fit_mds_with_stress(D, seed)
    r_ax = pidx // ncols
    c_ax = pidx % ncols
    ax = axs[r_ax, c_ax] if nrows > 1 else axs[c_ax]
    ax.scatter(coords[:, 0], coords[:, 1], s=18, color='k')
    # full mds
    for i, lab in enumerate(labels):
        ax.text(coords[i, 0], coords[i, 1], lab, fontsize=8, ha='left', va='center')
    ax.axhline(0, color='lightgray', linewidth=1)
    ax.axvline(0, color='lightgray', linewidth=1)
    ax.grid(True, alpha=0.2)
    ax.set_title(
        f"step {step_idx+1} (pos={pos:.2f})\n"
        f"(stress-1={stress1:.3f}, kstress={kst:.3f}, r={r:.3f})"
    )
    ax.set_xlabel('dimension 1')
    ax.set_ylabel('dimension 2')

# Remove any excess unused subplots
for k in range(nsteps, nrows*ncols):
    fig.delaxes(axs.flatten()[k])

plt.show()


In [ ]:
#Write b1s and b3s

import csv
import gzip
import shutil

# Extract training pair results and save to CSV
# We'll save the average margin over seeds for each pair (i, j) at each time step
# Additionally, save b1s and b3s activations (per time_step, item, unit) with per-seed values, mean, and std

# If results is a tuple, unpack it to get the actual results dict
if isinstance(results, tuple):
    results = results[1]
train_dict = results["train"]

# Margins (training_progress): shape (seeds_n, time_steps, items_n, items_n)
training_progress = train_dict["training_progress"]
seeds_n, time_steps, items_n, _ = training_progress.shape

# Prepare CSV header for margins
header_margins = ["time_step", "i", "j"] + [f"seed_{seed}" for seed in range(seeds_n)] + ["mean_margin", "std_margin"]

# Save b1s if available: shape (seeds_n, time_steps, items_n, b1_size)
if "b1s" in train_dict and train_dict["b1s"] is not None:
    b1s = train_dict["b1s"]
    b1_seeds, b1_time_steps, b1_items_n, b1_size = b1s.shape
    assert b1_seeds == seeds_n and b1_time_steps == time_steps and b1_items_n == items_n, "b1s shape mismatch"

    header_b1 = ["time_step", "item", "unit"] + [f"seed_{seed}" for seed in range(seeds_n)] + ["mean_activation", "std_activation"]
    rows_b1 = []
    for t in range(time_steps):
        for i in range(items_n):
            for unit in range(b1_size):
                vals = b1s[:, t, i, unit]
                rows_b1.append([t, i, unit] + list(vals) + [vals.mean(), vals.std()])

    b1_csv_filename = f"conjunctive_lazy_rich_b1s.csv"
    with open(b1_csv_filename, mode="w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(header_b1)
        writer.writerows(rows_b1)
    # Gzip the b1s CSV
    b1_csv_gz_filename = b1_csv_filename + ".gz"
    with open(b1_csv_filename, 'rb') as f_in, gzip.open(b1_csv_gz_filename, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    print(f"b1 activations saved to {b1_csv_filename} (gzipped to {b1_csv_gz_filename})")
else:
    print("b1s not found in results['train']; skipping b1 CSV.")

# Save b3s if available: shape (seeds_n, time_steps, items_n, b3_size)
if "b3s" in train_dict and train_dict["b3s"] is not None:
    b3s = train_dict["b3s"]
    b3_seeds, b3_time_steps, b3_items_n, b3_items_n = b3s.shape
    if b3_seeds == seeds_n and b3_time_steps == time_steps and b3_items_n == items_n:
        header_b3 = ["time_step", "item", "unit"] + [f"seed_{seed}" for seed in range(seeds_n)] + ["mean_activation", "std_activation"]
        rows_b3 = []
        for t in range(time_steps):
            for i in range(items_n):
                for unit in range(items_n):
                    vals = b3s[:, t, i, unit]
                    rows_b3.append([t, i, unit] + list(vals) + [vals.mean(), vals.std()])

        b3_csv_filename = f"conjunctive_lazy_rich_b3s.csv"
        with open(b3_csv_filename, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header_b3)
            writer.writerows(rows_b3)
        # Gzip the b3s CSV
        b3_csv_gz_filename = b3_csv_filename + ".gz"
        with open(b3_csv_filename, 'rb') as f_in, gzip.open(b3_csv_gz_filename, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        print(f"b3 activations saved to {b3_csv_filename} (gzipped to {b3_csv_gz_filename})")
    else:
        print("b3s shape mismatch with training_progress; skipping b3 CSV.")
else:
    print("b3s not found in results['train']; skipping b3 CSV.")